# AI Engineering Interview Prep
## Section: AI Agents & Agentic Systems

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


---
## 🤖 Section 4 — AI Agents and Agentic Systems

### Q1. What is an AI agent, and how does it differ from a simple LLM call?
**A:**
**Simple LLM call:** Send prompt → get response. One-shot. Static. You control everything.

**AI Agent:** The LLM can take **actions** — call tools, search the web, read/write files, execute code — and loops: think → act → observe → think → act again until the task is complete.

Key components of an agent:
- **Brain**: The LLM (reasoning engine)
- **Tools**: Functions the LLM can call (search, calculator, DB, APIs)
- **Memory**: Short-term (conversation) + Long-term (vector DB)
- **Agent loop**: The think → act → observe cycle

Analogy: LLM call = asking a smart friend a question. Agent = hiring a smart employee with a computer who figures out how to complete a multi-step project.

🏷️ *JobPilot AI: multi-agent CrewAI pipeline — 4 agents (resume analyst, JD analyser, resume writer, cover letter writer) each with specific tools and roles, collaborating to produce a complete application package.*


### Q2. What is AI Agent Memory?
**A:** Agent memory = how agents retain and recall information across steps and sessions.

**Short-term (In-context):** Conversation history within the current session's context window. Lost when session ends.

**Long-term (External):** Stored outside the model:
- **Episodic**: Records of past actions and their outcomes ("last time I searched X, I got Y")
- **Semantic**: Facts and knowledge stored in a vector DB or key-value store
- **Procedural**: Skills and how-to knowledge (stored in the system prompt or fine-tuning)

**Working memory**: Active information within the current context — retrieved facts, tool outputs, intermediate results.

🏷️ *Nyaya-Pro: short-term = last 5 turns in context window; long-term = per-user case history in PostgreSQL with vector embeddings for semantic retrieval of relevant past cases.*


### Q3. What is Harness Engineering in AI?
**A:** Harness engineering = building the infrastructure that safely houses, evaluates, and controls AI agents in production.

Like a safety harness for a climber — it doesn't limit capability, it prevents catastrophic failure.

Components:
- **Evaluation harness**: Automated test suites that run on every agent change; golden datasets, metrics
- **Sandbox harness**: Safe execution environment for agent actions (Docker sandbox for code, read-only file access)
- **Observability harness**: Logging, tracing, and monitoring every agent step, tool call, and token
- **Safety harness**: Guardrails, rate limits, human-in-the-loop checkpoints, irreversibility prevention
- **Cost harness**: Token budget tracking, cost alerts, automatic throttling when budget exceeded

Good harness engineering is what makes the difference between a demo agent and a production agent.


### Q4. Explain the ReAct (Reasoning + Acting) agent architecture.
**A:** ReAct interleaves reasoning and action in a loop:

```
Thought: What information do I need? I should search for bail provisions under BNS.
Action: search_legal_db(query="bail provisions murder BNS")
Observation: Section 437 BNS — bail in non-bailable offences...
Thought: I now have the relevant section. I can formulate my answer.
Final Answer: Under BNS Section 437...
```

Each Thought-Action-Observation triplet is one step. Continue until:
- LLM generates "Final Answer"
- Max steps reached
- Task validator confirms completion

Why effective: Grounds reasoning in actual tool outputs rather than pure hallucination. Each observation updates the model's understanding before the next reasoning step.

🏷️ *Nyaya-Pro's agent: Thought→identify legal domain→Action→route to FAISS index→Observation→retrieved sections→Thought→synthesise→Final Answer.*


### Q5. What is the Plan-and-Execute agent pattern?
**A:** Separates planning from execution:

**Phase 1 — Plan (expensive LLM):**
"Create a step-by-step plan to complete: {task}"
→ ["Step 1: Search for X", "Step 2: Analyse Y", "Step 3: Write Z", "Step 4: Review"]

**Phase 2 — Execute (cheaper model or tools):**
Execute each step sequentially or in parallel. If a step fails, the planner can revise the remaining plan.

Benefits:
- Expensive frontier model only used for planning (one call); cheaper models handle execution
- Full plan is visible before execution → easier to debug and audit
- Predictable, deterministic execution within each step
- Re-planning handles unexpected failures gracefully

Contrast with ReAct: ReAct decides one step at a time (reactive). Plan-and-Execute decides the full sequence upfront (proactive).

🏷️ *JobPilot AI: CrewAI manager agent creates the full pipeline plan (which agent does what); execution agents follow the plan sequentially.*


### Q6. What is tool use (function calling) in LLMs, and how does it enable agents?
**A:** Tool use (function calling) lets LLMs call external functions by generating structured JSON outputs that specify which function to call and with what arguments.

How it works:
1. Define available tools with schemas: name, description, parameters
2. Send tool schemas to the LLM alongside the prompt
3. LLM outputs: `{"tool": "search", "arguments": {"query": "BNS murder provisions"}}`
4. Your code executes the actual function
5. Return the result to the LLM as an "observation"
6. LLM reasons over the result and decides next action

Enables agents because: the LLM can interact with the real world (databases, APIs, files) rather than being limited to its training knowledge. This is how agentic behaviour emerges from an LLM.

Supported by: OpenAI (function calling), Google Gemini (function declarations), Anthropic Claude (tool use), Groq.


### Q7. How do you design and define tools for an AI agent?
**A:** Good tool design is critical — the LLM reads tool descriptions to decide which tool to use.

**Design principles:**
1. **Clear, specific names**: `search_legal_sections` not `search`
2. **Detailed description**: "Searches the Indian legal database (BNS, BNSS, Constitution) for sections matching the query. Returns relevant section text and citation."
3. **Typed parameters with descriptions**: Use JSON schema with `description` for each parameter
4. **Single responsibility**: Each tool does one thing well
5. **Predictable output format**: Consistent JSON structure the LLM can parse
6. **Error handling**: Tools should return structured errors the LLM can understand and react to

**Anti-patterns:**
- Too many tools → LLM picks wrong one. Keep to ≤10 tools per agent; group related tools.
- Vague descriptions → LLM misuses tools
- No parameter validation → agent passes garbage arguments


### Q8. What is the difference between single-agent and multi-agent systems?
**A:**
**Single-agent:** One LLM handles the entire task — reasoning, tool selection, execution, synthesis. Simple but: single point of failure, context grows without bound, can't parallelise.

**Multi-agent:** Multiple specialised agents collaborate:
- Each agent is optimised for its specific role
- Can run in parallel (faster)
- Failure of one agent doesn't fail the whole system
- Easier to debug (clear responsibility boundaries)
- More scalable — add agents for new capabilities

Trade-offs: More complex orchestration, inter-agent communication overhead, debugging is harder (need to trace across agents), more token consumption overall.

**When to use multi-agent:**
- Tasks that naturally decompose into parallel sub-tasks
- Different sub-tasks need different specialisations
- Single context window can't hold all the state
- Reliability is critical (redundant agents)

🏷️ *JobPilot AI: 4 agents in CrewAI — each is specialised, runs sequentially, passes structured output to the next. Much more reliable than one "do everything" agent.*


### Q9. What is Model Context Protocol (MCP), and how does it standardize tool integration?
**A:** MCP (Anthropic, 2024) is an open protocol for connecting LLMs to external tools and data sources. Think USB-C for AI — write a tool once as an MCP server, any MCP-compatible client can use it.

**Architecture:**
- **MCP Server**: Exposes resources (data), tools (functions), and prompts
- **MCP Client**: The AI application that connects to servers
- **Transport**: stdio (local) or HTTP/SSE (remote)

**Benefits:**
- Standardised — no custom integration code per LLM provider
- Composable — one server, many clients (Claude, any MCP client)
- Secure — explicit permission model for each capability
- Ecosystem — growing library of pre-built MCP servers (GitHub, Postgres, Slack, etc.)

**Without MCP:** each LLM app writes custom code to connect to each tool. With MCP: write the tool once as an MCP server → works with any compliant AI client.


### Q10. What are AI SubAgents?
**A:** SubAgents are specialised agents spawned by an orchestrator agent to handle specific subtasks.

Pattern:
```
Orchestrator: "Research X comprehensively"
├── SubAgent 1: "Search academic papers for X"
├── SubAgent 2: "Search recent news about X"
└── SubAgent 3: "Search code repositories for X implementations"
Orchestrator: synthesise all three results → final report
```

Benefits:
- **Parallelism** — all 3 SubAgents run simultaneously → faster
- **Specialisation** — each SubAgent has the right tools and context for its task
- **Scalability** — add more SubAgents to handle more sub-tasks
- **Isolation** — one SubAgent failing doesn't affect others

Implementation: LangGraph supports spawning parallel agent nodes; CrewAI supports crew hierarchy with sub-crews.

🏷️ *JobPilot AI: resume_analyst and jd_analyzer are effectively SubAgents running sequentially, their structured outputs feeding the writer agents.*


### Q11. What are the different types of agent memory (short-term, long-term, episodic)?
**A:**
| Memory Type | Storage | Duration | Use |
|-------------|---------|----------|-----|
| **Short-term** | Context window | Current session only | Current conversation, active task state |
| **Long-term semantic** | Vector DB / KV store | Persistent | Facts, knowledge, user preferences |
| **Episodic** | DB with timestamps | Persistent | Past actions and their outcomes; "what worked before" |
| **Procedural** | System prompt / fine-tuning | Permanent | How to do things; skills; agent persona |
| **Working** | Context window subset | Current task | Active variables, intermediate results |

In practice: most production agents use short-term (conversation window) + long-term semantic (vector DB for retrieval) + some form of episodic (logs of past tool calls and outcomes).


### Q12. How do you handle agent failures and implement error recovery?
**A:**
**Prevention:**
1. Validate tool inputs before execution
2. Test tools with edge cases independently
3. Implement tool timeouts

**Detection:**
1. Catch all exceptions from tool calls
2. Check tool output format (is it what was expected?)
3. Monitor for unexpected agent behaviours

**Recovery strategies:**
1. **Retry with backoff** — tool failed transiently? Retry 2-3 times
2. **Alternative tool** — if primary tool fails, fall back to alternative (e.g., if web search fails, try a different search API)
3. **Partial recovery** — if step 3 of 5 fails, can steps 1-2 still provide value? Return partial result.
4. **Graceful degradation** — if can't complete the task, return the best partial answer with an explanation of what failed
5. **Human escalation** — for critical failures, pause and alert a human operator
6. **Re-plan** — if the current plan is impossible, ask the planner to revise the approach


### Q13. What is an agent loop, and how does it decide when to stop?
**A:** The agent loop is the core execution cycle:

```python
while not done:
    thought = llm.think(system_prompt + history + observations)
    if thought.is_final_answer:
        return thought.answer
    
    action = thought.chosen_action
    observation = execute_tool(action)
    history.append(f"Action: {action}
Observation: {observation}")
    
    # Safety checks
    if steps > MAX_STEPS: raise MaxStepsError
    if total_tokens > TOKEN_BUDGET: raise BudgetError
    if time_elapsed > TIMEOUT: raise TimeoutError
```

**Stopping conditions:**
1. LLM generates "Final Answer" marker
2. Task validator confirms the goal is achieved
3. Max steps reached (hard safety limit — ALWAYS implement this)
4. Token budget exceeded
5. Timeout reached
6. Error recovery failed (agent is stuck)

Always implement max_steps. An agent without a stop condition is a runaway process.


### Q14. Context Engineering
**A:** Context engineering = the discipline of carefully designing what information goes into the LLM's context window at each step of an agent workflow.

It's broader than prompt engineering — it's about the entire information architecture:
- **What to include**: relevant memory, tool results, user state
- **What to exclude**: irrelevant history, redundant information
- **How to compress**: summaries of long tool outputs instead of full text
- **What order**: most important information first or last (lost in the middle)
- **What to retrieve**: which memories from long-term storage are relevant now

Poor context engineering: bloated context → high cost, confused agent, "lost in the middle" failures.
Good context engineering: lean, relevant context → lower cost, better reasoning, consistent behaviour.

🏷️ *Nyaya-Pro: I compress conversation history after 5 turns to a summary; retrieved legal sections are deduplicated and trimmed to the top-3 most relevant; tool outputs are formatted, not raw JSON.*


### Q15. How do AI Agents Communicate?
**A:** Several patterns for inter-agent communication:

**Sequential (pipeline):** Agent A's output → Agent B's input → Agent C. Simple assembly line.

**Shared state / blackboard:** All agents read/write to a shared state store. Flexible; needs coordination to avoid conflicts.

**Message passing:** Agents send structured messages to each other via a queue/bus. Decoupled; most robust for distributed systems.

**Hierarchical (manager-worker):** Orchestrator sends tasks to worker agents; workers return results; orchestrator synthesises.

**Parallel broadcast:** Orchestrator sends the same task to multiple agents; collects all responses; picks best or merges.

In practice (LangGraph / CrewAI): agents communicate via structured outputs (Pydantic models) that are validated before passing. The graph/workflow defines the communication topology.

🏷️ *CrewAI in JobPilot AI: sequential pipeline — each agent receives the previous agent's structured JSON output as its input context.*


### Q16. How do you evaluate and test AI agents?
**A:**
**Outcome metrics:**
- Task success rate (did it complete the goal?)
- Partial success rate (what % of sub-tasks completed?)

**Process metrics:**
- Steps taken (efficiency — fewer is better)
- Tool call accuracy (right tool with right parameters?)
- Token consumption per task
- Latency per task

**Trajectory evaluation:**
- For each step: was this the right action? (LLM-as-judge or human evaluation)
- Did the agent recover well from errors?
- Were there unnecessary steps?

**Test suite:**
- Build golden test cases with known correct outcomes
- Include edge cases: tool failures, ambiguous queries, multi-step reasoning
- Test regularly — agent quality can degrade when underlying LLM is updated

**Benchmarks:** WebArena, AgentBench, SWE-bench (coding agents), GAIA.


### Q17. What are the security risks of agentic systems, and how do you mitigate them?
**A:**
**Risks:**
1. **Prompt injection** — malicious content in tool outputs hijacks agent instructions
2. **Privilege escalation** — agent gains access to resources beyond its scope
3. **Data exfiltration** — agent sends sensitive data to external services
4. **Irreversible actions** — agent deletes data, sends emails, makes purchases without confirmation
5. **Tool abuse** — agent calls tools excessively (DoS) or with malicious arguments
6. **SSRF** — agent makes HTTP requests to internal services

**Mitigations:**
1. Principle of least privilege — tools have minimum necessary permissions
2. Sandbox all code execution (Docker, Firecracker)
3. Require human confirmation for irreversible actions
4. Input/output filtering on all tool calls
5. Rate limit tool calls
6. Audit log everything
7. Separate data plane from control plane in prompts


### Q18. What is the difference between reactive and proactive agents?
**A:**
**Reactive agents:** Respond to stimuli (user input, events). Take action only when triggered. Stateless between interactions. Simple, predictable.

Example: A customer support chatbot — answers when asked, does nothing otherwise.

**Proactive agents:** Monitor for conditions, take initiative without being asked. Maintain state, operate in the background.

Example: An agent that monitors your inbox, summarises important emails, and alerts you proactively. Or an agent that detects a production anomaly and initiates investigation before you notice.

**Trade-offs:**
- Reactive: safer (limited scope), easier to debug, user-controlled
- Proactive: more powerful, but harder to control, potential for unwanted actions

Most production agents are reactive (user-triggered). Proactive agents need very careful guardrails — they can take expensive or irreversible actions without human initiation.


### Q19. How do you manage token consumption and cost in long-running agent workflows?
**A:**
1. **Token budget per task** — set a hard limit (e.g., 50K tokens); agent self-reports progress; fails gracefully at limit
2. **Context compression** — summarise old tool outputs and conversation turns instead of appending full text
3. **Smaller model for simple steps** — use GPT-4o-mini for tool selection and simple reasoning; GPT-4o only for complex synthesis
4. **Tool output truncation** — don't pass 10K-token web pages into context; extract and summarise the relevant 200 tokens
5. **Caching** — cache common tool call results; if web search for "BNS murder provisions" was called 5 times this hour, cache it
6. **Plan-and-execute** — expensive model for the plan (one call), cheap model for each execution step
7. **Monitor and alert** — set cost alerts per task type; investigate outliers

🏷️ *JobPilot AI: each CrewAI agent has a max_iteration limit and token budget. Tool outputs (resume parse, JD analysis) are summarised by the ingestion step before passing to writer agents.*


### Q20. What is the human-in-the-loop (HITL) pattern for agents, and when is it needed?
**A:** HITL pauses agent execution at certain checkpoints to get human approval before proceeding.

**When it's needed:**
- Before **irreversible actions** (send email, delete data, make payment, push to production)
- When **agent confidence is low** (uncertainty detected in reasoning)
- For **high-stakes decisions** (medical, legal, financial)
- **Regulatory compliance** — some regulated industries require human sign-off on AI decisions
- When **novel situations** outside the agent's training distribution arise

**Implementation:**
1. Define a set of "checkpoint actions" that require human approval
2. Agent generates an "approval request" with the planned action and reasoning
3. Human reviews and approves/rejects/modifies
4. Agent continues from human decision
5. Log every HITL decision for audit

LangGraph supports HITL natively with `interrupt_before` and `interrupt_after` node hooks.


### Q21. How do you implement guardrails for AI agents to prevent harmful actions?
**A:**
**Input guardrails** (before agent acts):
1. Input classifier — detect off-topic, harmful, or injection attempts
2. Intent validator — does this request match the agent's scope?

**Action guardrails** (before tool execution):
1. Allowlist of permitted tool calls per agent role
2. Parameter validation — check tool arguments for sanity before execution
3. Irreversibility check — if action is irreversible, require HITL
4. Rate limiting — cap tool calls per minute

**Output guardrails** (before returning to user):
1. Content filter — check for harmful, offensive, or PII content
2. Topic compliance — is the response on-topic for this agent's domain?
3. Format validation — does the output match the expected schema?

Tools: NeMo Guardrails (NVIDIA), Guardrails AI, LLM Guard, custom rule-based validators.


### Q22. What is agent reflection, and how does it improve agent performance?
**A:** Reflection = the agent evaluates its own previous outputs and actions to identify errors and improve.

**Reflexion pattern:**
1. Agent produces output X
2. Reflection step: "Critique this output. What's correct? What's wrong or incomplete? What would you do differently?"
3. Agent revises based on the critique
4. Repeat until quality threshold met or max iterations

**Why it improves quality:** The agent acts as its own code reviewer. Catches logical errors, format issues, incomplete answers. Adds 1-2 extra LLM calls but significantly improves quality for complex tasks.

**Where to use:** Code generation and review, long-form writing, complex reasoning tasks. Not worth the cost for simple Q&A.

🏷️ *JobPilot AI's cover letter writer has a reflection step — after generation, it self-critiques: "Does this cover letter address the top 3 JD requirements?" and revises if not.*


### Q23. What is the difference between code-generating and tool-calling agents?
**A:**
**Tool-calling agents:** Use pre-defined, fixed tools (search, calculator, DB query). The LLM selects from a catalogue and calls with arguments. Safe, predictable, easy to audit. Limited to what the tool catalogue offers.

**Code-generating agents:** Write and execute arbitrary code to accomplish tasks. Infinite flexibility — if you can code it, the agent can do it. But: requires a secure sandbox, risk of malicious/buggy code, output is less predictable.

**Trade-offs:**
| | Tool-calling | Code-generating |
|--|-------------|----------------|
| Safety | High | Requires sandboxing |
| Flexibility | Limited to tools | Unlimited |
| Predictability | High | Lower |
| Debugging | Easy (discrete steps) | Harder |
| Best for | Well-defined tasks | Novel, complex data processing |

Most production agents use tool-calling. Code generation is used in specialised apps (GitHub Copilot Workspace, Devin, code interpreters).


### Q24. How do you handle multi-modal inputs and outputs in agentic systems?
**A:**
**Inputs:** Vision-language models (GPT-4V, Gemini) can process images alongside text in the agent's context. Pass image URLs or base64-encoded images in the messages array.

**Outputs:** Agents can generate images (via DALL-E tool call), audio (TTS tool), or structured data visualisations.

**Design considerations:**
1. Image inputs: define `image_input` tool that takes a URL and returns a text description (if using a text-only orchestrator)
2. Audio inputs: transcribe with Whisper first, pass transcript to agent
3. Multi-modal memory: store image embeddings (CLIP) in vector DB for retrieval
4. Output routing: if user needs an image, agent calls an image generation tool and returns the URL

🏷️ *ExamGenie AI: Gemini 2.5 Flash natively handles PDF page images — the agent receives the image and question generation request, processes both modalities in one call.*


### Q25. How do you implement state management in complex agent workflows?
**A:**
**What to manage:**
- Current task and sub-tasks
- Tool call results (intermediate outputs)
- Agent decisions and reasoning
- Conversation history
- User preferences / session data

**Approaches:**
1. **In-context state** — pass all state as part of the context. Simple but limited by context window.
2. **External state store** — Redis/PostgreSQL holds the full workflow state. Agent reads/writes via tool calls. Enables pausing/resuming.
3. **LangGraph** — nodes are agent steps; edges are transitions; graph state is explicitly typed (TypedDict); checkpointing via LangSmith.
4. **Event sourcing** — record every agent action as an immutable event; reconstruct state from event log.

For production: use LangGraph with Postgres/Redis checkpointer — supports pause/resume, human-in-the-loop, distributed execution, and full state history for debugging.


### Q26. How do you build a customer support agent with escalation logic?
**A:**
**Architecture:**
```
User Message
→ Intent Classifier
→ Tier 1 FAQ Agent (70% of queries) → Answer + resolve
→ Tier 2 Account Agent (needs CRM) → lookup + resolve
→ Tier 3 Technical Agent (complex) → KB search + resolve
→ Tier 4 Human Escalation (sensitive/angry/unresolved)
```

**Escalation triggers:**
- Sentiment < threshold (user is angry)
- Topic matches escalation keywords ("cancel subscription", "legal action")
- Agent confidence < threshold after 2 attempts
- User explicitly requests human
- Safety issue detected (medical emergency, abuse)

**Implementation:**
- Intent classifier routes the initial request
- Each tier has a max_attempts limit
- Fallback chain: Tier N → Tier N+1 on failure or trigger
- Human handoff: creates a ticket with full conversation context for the human agent
- CSAT tracking: collect feedback after resolution

🏷️ *This is exactly what I'd build for Yotta's enterprise customer support — tier-based routing by technical complexity, with Keycloak-authenticated escalation to human agents.*


### Q27. What is agent orchestration, and how do you implement it?
**A:** Orchestration = managing multiple agents — routing tasks, handling dependencies, managing shared state, dealing with failures.

**Key responsibilities:**
- **Routing**: which agent handles this task/step?
- **Scheduling**: sequential vs parallel execution
- **State management**: shared context all agents can access
- **Error handling**: if Agent B fails, does Agent A's work get rolled back?
- **Result aggregation**: how to combine outputs from multiple agents

**Tools:**
- **LangGraph**: graph-based, explicit state, great for conditional flows and HITL
- **CrewAI**: role-based, declarative, higher-level abstraction
- **AutoGen**: conversation-based, agents communicate via messages
- **Custom**: for simple linear pipelines, custom Python orchestration is often clearest

🏷️ *CrewAI in JobPilot AI handles orchestration automatically — task sequencing, agent communication, output passing. For Nyaya-Pro's conditional routing, I use custom Python because the conditions are simple enough.*


### Q28. How do you build a code execution agent safely using sandboxed environments?
**A:**
**The risk**: A code-generating agent running on the host system can delete files, make network calls, consume all CPU, install malware.

**Safe sandbox design:**
1. **Docker container** — isolated filesystem, no host access
2. **Resource limits**: `--memory 512m --cpus 0.5 --pids-limit 50`
3. **Network isolation**: `--network none` — no internet access
4. **Read-only filesystem** except a `/workspace` volume
5. **Non-root user** inside the container
6. **Timeout**: kill container after 30 seconds
7. **Allowlist** of permitted system calls (seccomp profile)
8. **Ephemeral**: destroy container after each execution

**Output handling**: Return stdout, stderr, exit code, and any files written to `/workspace`.

**Tools**: E2B (managed sandboxes), Modal, AWS Lambda with VPC isolation.


### Q29. Your AI agent is stuck in an infinite loop. How do you detect and break the cycle?
**A:**
**Prevention (build in from the start):**
1. `max_steps` hard limit — raise MaxStepsError after N iterations
2. State hashing — hash (action + args) at each step; if seen before, break
3. Action deduplication — if the same tool is called with the same args 3× in a row, break
4. Progress validator — after each step, check if progress is being made toward the goal

**Detection in production:**
1. Monitor step count per task — alert if > 10 steps for a task that normally takes 3
2. Monitor token count — runaway loops consume tokens linearly
3. Watch for repeated tool calls in logs

**Recovery:**
1. Add instruction: "If you've tried the same approach more than twice without success, stop and explain what you've tried and why you're stuck."
2. Inject: "You are in a loop. The last 3 actions were identical. Try a completely different approach or admit you cannot solve this."
3. Kill the agent task and return a partial result with an explanation


### Q30. Your AI agent gets conflicting answers from different tools. How does it reconcile them?
**A:**
1. **Explicit reconciliation prompt**: "Tool A returned X and Tool B returned Y. These contradict. Explain which is more likely correct and why."
2. **Source ranking** — define a priority hierarchy: official DB > web search > cached data. Prefer higher-ranked source.
3. **Recency preference** — prefer the most recently updated source (add timestamps to tool outputs)
4. **Confidence scoring** — some tools return confidence scores; prefer the higher-confidence answer
5. **Seek a third source** — if two tools conflict, call a third authoritative source
6. **Aggregate and flag** — return both answers to the user, explain the conflict, let the user decide
7. **Validation tool** — have a dedicated "fact check" tool that independently verifies a claim

Design principle: the agent should never silently pick one conflicting answer. It should always surface the conflict to the user or orchestrator.


### Q31. Your AI agent burns too many tokens per task. How do you reduce token consumption?
**A:**
1. **Summarise, don't append** — compress old tool outputs and conversation turns into brief summaries; don't keep growing the context
2. **Model routing** — use GPT-4o-mini for simple reasoning steps; GPT-4o only for complex synthesis
3. **Trim tool outputs** — web search returns 5000 tokens? Extract relevant 200 tokens before adding to context
4. **Shorten system prompts** — remove verbose instructions; use concise imperative statements
5. **Plan-and-execute** — one expensive planning call + many cheap execution calls
6. **Cache tool results** — repeated tool calls with the same arguments should return cached results
7. **Set strict max_tokens** — force shorter responses; the model often over-explains
8. **Reduce agent steps** — combine steps that can be done in one call instead of two

🏷️ *JobPilot AI: tool outputs (resume JSON, JD keywords) are summarised before passing to writer agents — saved ~40% tokens vs raw output passing.*


### Q32. Your AI agent keeps exceeding its budget per task. How do you enforce budget limits?
**A:**
1. **Token counter wrapper** — wrap every LLM call to track cumulative tokens; raise BudgetExceeded when limit hit
2. **Cost estimation before execution** — estimate the cost of the planned steps; abort if estimated cost > budget
3. **Progressive budget** — allocate a small initial budget; agent can request more if needed (with justification)
4. **Step-level budgets** — each step has its own token limit; agent can't exceed per-step limit
5. **Graceful degradation** — when budget is 80% consumed, switch to cheaper model; when 95%, return best partial result
6. **Alert system** — notify on budget anomalies; investigate tasks that consistently exceed budget
7. **Cost dashboard** — visible per-task, per-user cost tracking; identify expensive query types
8. **Model tier routing** — automatically downgrade to cheaper model as budget shrinks


### Q33. Your AI agent hallucinates tool capabilities and passes wrong inputs. How do you fix it?
**A:** The agent invents non-existent tool parameters or misunderstands what a tool does.

**Fixes:**
1. **Better tool descriptions** — the most important fix. Make descriptions extremely specific: what the tool does, what it does NOT do, valid parameter values, expected output format
2. **Parameter validation** — validate tool arguments before execution; return a clear error message that the agent can learn from
3. **Typed schemas** — use strict JSON schema with enum values, regex patterns, min/max constraints
4. **Few-shot tool examples** — include correct usage examples in the tool description
5. **Tool testing** — if an invalid argument is passed, return: "Error: `query` must be a string. You passed: `{'query': null}`. Please retry with a valid string."
6. **Reduce tool count** — fewer tools = lower chance of confusion. Group related functionality.
7. **Fine-tune on tool use** — fine-tune the model on your specific tool schemas for consistent usage


### Q34. Your AI agent deleted a production database. How do you prevent irreversible actions?
**A:** This is the most critical safety problem in agentic systems.

**Prevention architecture:**
1. **Classify action reversibility** — every tool is tagged: `reversible=True/False`
2. **HITL for irreversible actions** — agent generates a "I intend to do X" request; human must approve before execution
3. **Dry-run mode** — all write/delete operations have a `dry_run=True` mode that simulates without executing
4. **Read-only agents by default** — agents start with read-only permissions; write permissions must be explicitly requested
5. **Soft deletes** — never hard-delete; mark as deleted with timestamp; allow recovery
6. **Audit log** — log every action with timestamp, agent ID, action, arguments — immutable append-only log
7. **Rate limiting** — limit destructive operations per hour
8. **Staging environment first** — test all new agent behaviours in staging before production

Post-incident: always investigate HOW the agent reached the decision to delete. Was it a prompt injection? A misunderstood tool description? Fix the root cause.


### Q35. Your AI agent has many tools but keeps picking the wrong one. How do you improve tool selection?
**A:**
1. **Better tool descriptions** — each description should make it crystal clear what the tool does AND what it doesn't do
2. **Reduce tool count** — prune tools the agent rarely needs; more tools = more selection confusion. Keep ≤10 per agent.
3. **Tool grouping** — group related tools and present them in logical categories with a brief category description
4. **Few-shot tool selection examples** — in the system prompt, show 2-3 examples of correct tool selection for representative queries
5. **Two-step selection** — step 1: classify the task type; step 2: within that category, select the specific tool
6. **Dedicated router agent** — have a small, specialised "tool selector" LLM that only decides which tool to use; larger model only executes
7. **Dynamic tool injection** — only inject the most relevant tools for the current task type (based on query classification) rather than all tools every time


### Q36. Your AI agent takes too long to complete a task. How do you speed it up?
**A:**
1. **Parallelise independent steps** — if steps A and B don't depend on each other, run them simultaneously (async/threading)
2. **Smaller, faster model** — use GPT-4o-mini or Groq (Llama-3.3-70b at 400+ tokens/sec) instead of GPT-4o for most steps
3. **Reduce unnecessary steps** — audit the agent trajectory; are any steps redundant? Remove them.
4. **Cache tool results** — if a tool call's result won't change, cache it for the session duration
5. **Stream responses** — don't wait for full completion; start processing partial results
6. **Plan-and-execute** — clear upfront plan avoids backtracking and wasted steps
7. **Reduce context** — smaller context = faster prefill = lower TTFT per step
8. **Async tool calls** — while one tool executes (e.g., web search), start preparing the next step's context

🏷️ *Groq API (Llama-3.3-70b) in JobPilot AI: 400+ tokens/sec vs ~50 tokens/sec for GPT-4o — the entire pipeline runs in ~8 seconds vs ~40 seconds with OpenAI.*


### Q37. Your LLM selects the right tool but extracts the wrong parameters. How do you fix parameter extraction?
**A:**
1. **Improve parameter descriptions** — each parameter needs a clear description AND an example value
2. **Add constraints to schema** — use JSON Schema features: `enum` (allowed values), `pattern` (regex), `minimum`/`maximum`, `required`
3. **Validation feedback loop** — validate extracted parameters before calling the tool; if invalid, return a structured error with the expected format: "Parameter 'date' received '20-06-2026'. Expected ISO 8601 format: '2026-06-20'. Please retry."
4. **Few-shot parameter examples** — in the tool description, show: "Example call: `search_legal_db(query='bail provisions murder', domain='Criminal')`"
5. **Type coercion** — add a normalisation layer that converts common formats (e.g., string "true" → boolean True)
6. **Reduce parameter count** — fewer parameters = fewer extraction errors. Combine related parameters or use sensible defaults.
7. **Fine-tune on tool use data** — if systematic errors exist, fine-tune on correct (query, tool_call) pairs
